**BERT embeddings → UMAP → DBSCAN → cluster-aware denoising autoencoder**

Embed `products`, `adj`, and `verb` separately with `bert-base-uncased`.
The CSV spelling `product` is also accepted. Both datasets share one UMAP projection and are clustered
together so their cluster IDs refer to the same groups.

Each column embedding is a unit-normalized mean of its BERT token vectors.
The row embedding is **0.6 × products + 0.2 × adj + 0.2 × verb**,
giving products greater weight while treating adjectives and verbs equally.
Missing columns raise an error; empty fields contribute a zero vector.
Weights are fixed, even when a field is empty.

A cluster's average embedding is the arithmetic mean of its members'
weighted row embeddings. Noise (cluster `-1`) has no cluster average.
Calories and other numeric columns are not included in these embeddings.

Run the installation cell below before the imports to install all required libraries
into this notebook's kernel.
The first model load downloads the pretrained weights from Hugging Face.

In [ ]:
# CUDA 12.6 builds support this environment's Tesla V100 (CUDA 13 builds do not).
# Keep all three compiled packages on the same CUDA build.
%pip install transformers==5.17.0 numpy pandas scikit-learn scipy umap-learn ipython torch torchvision torchaudio

# Once installation succeeds, restart the kernel, then run the verification cell below.


Looking in indexes: https://download.pytorch.org/whl/cu126
ERROR: Could not find a version that satisfies the requirement torch==2.14.0+cu126 (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
ERROR: No matching distribution found for torch==2.14.0+cu126
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 31.8 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 30.2 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uni

**After installing: restart the kernel, then run the cells below.**

Use **Kernel → Restart Kernel** (or your editor's Restart command). This clears
Transformers' cached backend detection and previously imported package versions.
The verification cell prints the active Python interpreter and library versions.
If installation fails, fix that error before continuing.

The pip package is `torch`. For a specific CUDA/ROCm build, use the command matching
your hardware from the [official PyTorch installer](https://pytorch.org/get-started/locally/)
in this notebook's kernel before restarting.

In [ ]:
import ast
import json
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy import sparse
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors, sort_graph_by_row_values
from sklearn.preprocessing import normalize
from transformers import AutoModel, AutoTokenizer
from umap import UMAP

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Disabling PyTorch because PyTorch >= 2.5 is required but found 2.2.1


In [ ]:
DATA_DIR = Path("input_stage2")
dataset_paths = {
    "new_dataset_recommendation": DATA_DIR / "recipe_user_ratings.csv",
    "current_dataset": DATA_DIR / "price_mapped_nutrients.csv",
}
missing_paths = [str(path) for path in dataset_paths.values() if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Restore the missing CSVs or fix dataset_paths before running: "
        + ", ".join(missing_paths)
        + ". Broken symlinks also count as missing files."
    )

new_dataset_recommendation_df = pd.read_csv(dataset_paths["new_dataset_recommendation"])
current_dataset_df = pd.read_csv(dataset_paths["current_dataset"])

In [ ]:
display(new_dataset_recommendation_df.head())
display(current_dataset_df.head())

,recipe_id,name,calories (g),total fat (g),sugar (g),sodium (g),protein (g),saturated fat (g),carbohydrates (g),ingredients,user_id,rating,product,adj,verb,price_breakdown
0,137739,arriba baked winter squash mexican style,103000.0,0.0,650.0,0.0,100.0,0.0,1200.0,"['winter squash', 'mexican seasoning', 'mixed ...",4470,5,"[[""winter squash""], [""mexican seasoning""], [""m...","[[], [], [], [], [], [], []]","[[], [], [], [], [], [], []]","{""mixed spice"": null, ""olive oil"": 19.9, ""wint..."
1,137739,arriba baked winter squash mexican style,103000.0,0.0,650.0,0.0,100.0,0.0,1200.0,"['winter squash', 'mexican seasoning', 'mixed ...",593927,5,"[[""winter squash""], [""mexican seasoning""], [""m...","[[], [], [], [], [], [], []]","[[], [], [], [], [], [], []]","{""mixed spice"": null, ""olive oil"": 19.9, ""wint..."
2,137739,arriba baked winter squash mexican style,103000.0,0.0,650.0,0.0,100.0,0.0,1200.0,"['winter squash', 'mexican seasoning', 'mixed ...",178427,5,"[[""winter squash""], [""mexican seasoning""], [""m...","[[], [], [], [], [], [], []]","[[], [], [], [], [], [], []]","{""mixed spice"": null, ""olive oil"": 19.9, ""wint..."
3,31490,a bit different breakfast pizza,346800.0,1170.0,0.0,40800.0,1100.0,700.0,300.0,"['prepared pizza crust', 'sausage patty', 'egg...",319943,4,"[[""pizza""], [""sausage patty""], [""egg""], [""milk...","[[], [], [], [], [], []]","[[""prepared""], [], [], [], [], []]","{""milk"": 3.63, ""prepared pizza crust"": 9.64, ""..."
4,31490,a bit different breakfast pizza,346800.0,1170.0,0.0,40800.0,1100.0,700.0,300.0,"['prepared pizza crust', 'sausage patty', 'egg...",674022,5,"[[""pizza""], [""sausage patty""], [""egg""], [""milk...","[[], [], [], [], [], []]","[[""prepared""], [], [], [], [], []]","{""milk"": 3.63, ""prepared pizza crust"": 9.64, ""..."


,fdc_id,description,product,item_prices,unpriced_items,priced_items,total_items,priced_share,adj,verb,total_price,protein_g,fat_g,carb_g,energy_kcal,fiber_g,sodium_mg,cholesterol_mg,satfat_g,sugars_g
0,167512,"Pillsbury Golden Layer Buttermilk Biscuits, Ar...","['layer buttermilk biscuits', 'dough']",{'layer buttermilk biscuits': 4.5},['dough'],1,2,0.5,"['Golden', 'Artificial']",['refrigerated'],0.00,5.88,13.24,41.18,307.0,1.2,1059.0,0.0,2.941,5.88
1,167516,"Waffles, buttermilk, frozen, ready-to-heat",['waffles'],{'waffles': 6.85},[],1,1,1.0,['ready'],['frozen'],6.85,6.58,9.22,41.05,273.0,2.2,621.0,15.0,1.898,4.30
2,167517,"Waffle, buttermilk, frozen, ready-to-heat, toa...",['waffle'],{'waffle': 6.85},[],1,1,1.0,['ready'],"['frozen', 'toasted']",6.85,7.42,9.49,48.39,309.0,2.6,710.0,13.0,2.275,4.41
3,167518,"Waffle, buttermilk, frozen, ready-to-heat, mic...",['waffle'],{'waffle': 6.85},[],1,1,1.0,['ready'],"['frozen', 'microwaved']",6.85,6.92,9.40,44.16,289.0,2.4,663.0,16.0,2.057,4.50
4,167519,"Waffle, plain, frozen, ready-to-heat, microwave",['waffle'],{'waffle': 6.85},[],1,1,1.0,"['plain', 'ready']",['frozen'],6.85,6.71,9.91,45.41,298.0,2.4,682.0,16.0,1.580,5.04


In [ ]:
MODEL_NAME = "bert-base-uncased"
FEATURE_COLUMNS = ("products", "adj", "verb")
COLUMN_WEIGHTS = {"products": 0.6, "adj": 0.2, "verb": 0.2}
BERT_BATCH_SIZE = 32
MAX_LENGTH = 512  # Longer fields are truncated at BERT's token limit.

# Starting values: tune EPS and MIN_SAMPLES after inspecting cluster sizes/noise.
EPS = 0.5  # Euclidean radius in UMAP space; tune after inspecting clusters.
MIN_SAMPLES = 5  # Counts original rows, including repeated user ratings.
GRAPH_BATCH_SIZE = 2048
MAX_GRAPH_EDGES = 10_000_000

weights = np.array([COLUMN_WEIGHTS[column] for column in FEATURE_COLUMNS])
if not (
    np.isfinite(weights).all()
    and weights[0] > weights[1] > 0
    and weights[1] == weights[2]
    and np.isclose(weights.sum(), 1.0)
):
    raise ValueError("Use positive weights summing to 1 with products > adj = verb.")

RANDOM_STATE = 42
UMAP_COMPONENTS = 10
UMAP_NEIGHBORS = 30
UMAP_MIN_DIST = 0.0


**Prepare the text and share repeated embeddings**

Nested JSON/Python lists such as `[["olive oil"], ["salt"]]` become
`olive oil; salt`; `[[], []]` becomes an empty string. The original columns
and dataframe row order are preserved. Identical feature combinations
share one weighted embedding, and identical field texts are encoded once.

In [ ]:
@lru_cache(maxsize=200_000)
def clean_string_feature(value):
    value = value.strip()
    if value.lower() in {"", "nan", "none", "null", "<na>"}:
        return ""
    if value.startswith(("[", "(")):
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            parsed = ast.literal_eval(value)
        return feature_to_text(parsed)
    return " ".join(value.lower().split())


def feature_to_text(value):
    if isinstance(value, str):
        return clean_string_feature(value)
    if isinstance(value, (list, tuple, np.ndarray)):
        terms = [feature_to_text(item) for item in value]
        return "; ".join(term for term in terms if term)
    if value is None or pd.isna(value):
        return ""
    raise TypeError(f"Expected text or a list of terms, got {type(value).__name__}.")


def prepare_feature_texts(df):
    product_column = "products" if "products" in df.columns else "product"
    source_columns = {"products": product_column, "adj": "adj", "verb": "verb"}
    missing = [column for column in source_columns.values() if column not in df.columns]
    if missing:
        raise ValueError(f"Missing embedding columns: {missing}")
    return pd.DataFrame({
        feature: df[source_column].map(feature_to_text).to_numpy()
        for feature, source_column in source_columns.items()
    })


datasets = {
    "new_dataset_recommendation": new_dataset_recommendation_df,
    "current_dataset": current_dataset_df,
}
combined_features = pd.concat(
    [prepare_feature_texts(df) for df in datasets.values()], ignore_index=True
)
if combined_features.empty:
    raise ValueError("Both datasets are empty; there are no rows to cluster.")

# Positional IDs preserve duplicate/non-contiguous dataframe indexes.
row_embedding_ids, unique_keys = pd.factorize(
    pd.MultiIndex.from_frame(combined_features), sort=False
)
unique_features = unique_keys.to_frame(index=False)
unique_features.columns = FEATURE_COLUMNS
row_counts = np.bincount(row_embedding_ids, minlength=len(unique_features))
unique_texts = pd.Index(pd.unique(unique_features.to_numpy().ravel()))
feature_text_ids = {
    column: unique_texts.get_indexer(unique_features[column])
    for column in FEATURE_COLUMNS
}
clean_string_feature.cache_clear()
del combined_features, unique_keys
print(
    f"{len(row_embedding_ids):,} rows; {len(unique_features):,} unique feature "
    f"combinations; {len(unique_texts):,} distinct field texts."
)

1,143,212 rows; 238,165 unique feature combinations; 284,884 distinct field texts.


**Create BERT embeddings and the weighted sum**

Mean pooling uses the final hidden layer and excludes padding and special
tokens. Inference runs in evaluation mode, on CUDA when available.
Each dataframe receives `products_embedding`, `adj_embedding`,
`verb_embedding`, and `weighted_embedding` columns below. Repeated values
reference shared arrays, so treat those vectors as read-only.

In [ ]:
# A detected GPU may still be unavailable to this runtime; test an actual operation.
device = torch.device("cpu")
if torch.cuda.is_available():
    try:
        _cuda_probe = torch.ones(1, device="cuda") + 1
        torch.cuda.synchronize()
        del _cuda_probe
        device = torch.device("cuda")
    except RuntimeError as exc:
        print(f"CUDA cannot execute in this runtime; using CPU. Details: {exc}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert_model.eval()
print(f"Embedding with {MODEL_NAME} on {device}.")


def embed_texts(texts, batch_size=BERT_BATCH_SIZE):
    if batch_size < 1:
        raise ValueError("batch_size must be positive.")
    embeddings = np.zeros((len(texts), bert_model.config.hidden_size), dtype=np.float32)
    nonempty_ids = [index for index, text in enumerate(texts) if text]
    with torch.inference_mode():
        for start in range(0, len(nonempty_ids), batch_size):
            ids = nonempty_ids[start:start + batch_size]
            inputs = tokenizer(
                [texts[index] for index in ids],
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_special_tokens_mask=True,
                return_tensors="pt",
            )
            special_mask = inputs.pop("special_tokens_mask").to(device)
            inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
            hidden = bert_model(**inputs).last_hidden_state
            token_mask = (inputs["attention_mask"].bool() & ~special_mask.bool())
            token_mask = token_mask.unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * token_mask).sum(dim=1) / token_mask.sum(dim=1).clamp(min=1)
            embeddings[ids] = pooled.cpu().numpy()
    return normalize(embeddings, norm="l2", copy=False) if len(texts) else embeddings


text_embeddings = embed_texts(unique_texts.tolist())
weighted_embeddings = np.zeros(
    (len(unique_features), bert_model.config.hidden_size), dtype=np.float32
)
for column, weight in COLUMN_WEIGHTS.items():
    for start in range(0, len(unique_features), 4096):
        stop = start + 4096
        weighted_embeddings[start:stop] += (
            weight * text_embeddings[feature_text_ids[column][start:stop]]
        )

# Store shared vector references instead of copying 768 floats per repeated row.
text_embedding_rows = pd.Series(list(text_embeddings), dtype=object)
weighted_embedding_rows = pd.Series(list(weighted_embeddings), dtype=object)
offset = 0
for df in datasets.values():
    ids = row_embedding_ids[offset:offset + len(df)]
    df["embedding_row_id"] = ids
    for column in FEATURE_COLUMNS:
        text_ids = feature_text_ids[column][ids]
        df[f"{column}_embedding"] = text_embedding_rows.iloc[text_ids].to_numpy()
    df["weighted_embedding"] = weighted_embedding_rows.iloc[ids].to_numpy()
    offset += len(df)

print(f"Weighted embedding matrix: {weighted_embeddings.shape}")

**Reduce dimensions with UMAP, then cluster both datasets**

Fit one UMAP model on the nonempty, unique weighted BERT vectors from both
datasets. Use cosine distance as UMAP's input metric, 10 output dimensions,
30 neighbors, and `min_dist=0.0`. DBSCAN then uses **Euclidean distance in
this shared reduced space**, so its `EPS` has different units from cosine
distance. The seed is fixed. UMAP is fitted without ratings or user IDs.

Duplicate feature combinations carry their original row counts as DBSCAN
`sample_weight`. UMAP itself is fitted on unique combinations without these
multiplicities. A frequently rated recipe can meet `MIN_SAMPLES` by itself.
Zero BERT vectors are excluded and remain noise (`-1`).

A sparse radius graph is built in batches; a tree search can use the lower
dimensional coordinates. `MAX_GRAPH_EDGES` limits accumulated graph size.
UMAP changes local density, so inspect cluster sizes/noise and tune `EPS`,
`MIN_SAMPLES`, and UMAP parameters. Defaults are not calibrated quality claims.
Cluster averages and recommender features remain in the original weighted
BERT space, preserving their meaning independently of UMAP coordinates.

In [ ]:
valid_embedding_ids = np.flatnonzero(np.linalg.norm(weighted_embeddings, axis=1) > 0)
# NaN denotes a missing embedding; the coordinate zero itself is a valid UMAP point.
reduced_embeddings = np.full((len(weighted_embeddings), UMAP_COMPONENTS), np.nan, dtype=np.float32)
umap_model = None
if len(valid_embedding_ids):
    if len(valid_embedding_ids) < 3:
        raise ValueError("UMAP needs at least three nonempty unique feature combinations.")
    umap_model = UMAP(
        n_components=UMAP_COMPONENTS,
        n_neighbors=min(UMAP_NEIGHBORS, len(valid_embedding_ids) - 1),
        min_dist=UMAP_MIN_DIST,
        metric="cosine",
        init="random",  # Also works when a small sample has fewer rows than output dimensions.
        random_state=RANDOM_STATE,
        n_jobs=1,
        low_memory=True,
    )
    reduced_embeddings[valid_embedding_ids] = umap_model.fit_transform(
        weighted_embeddings[valid_embedding_ids]
    )
    if not np.isfinite(reduced_embeddings[valid_embedding_ids]).all():
        raise ValueError("UMAP produced non-finite coordinates. Inspect the inputs and neighbor settings.")

reduced_embedding_rows = pd.Series(list(reduced_embeddings), dtype=object)
for df in datasets.values():
    df["umap_embedding"] = reduced_embedding_rows.iloc[df["embedding_row_id"].to_numpy()].to_numpy()
print(f"UMAP dimensions: {weighted_embeddings.shape[1]} → {UMAP_COMPONENTS}")

In [ ]:
def cluster_weighted_embeddings(
    embeddings, counts, eps=EPS, min_samples=MIN_SAMPLES,
    batch_size=GRAPH_BATCH_SIZE, max_graph_edges=MAX_GRAPH_EDGES,
):
    if not np.isfinite(eps) or eps <= 0 or min_samples < 1 or batch_size < 1:
        raise ValueError("eps, min_samples, and batch_size must be positive.")
    labels = np.full(len(embeddings), -1, dtype=np.int64)
    valid_ids = np.flatnonzero(np.isfinite(embeddings).all(axis=1))
    if not len(valid_ids):
        return labels, None

    valid_embeddings = embeddings[valid_ids]
    neighbors = NearestNeighbors(radius=eps, metric="euclidean", algorithm="auto", n_jobs=-1)
    neighbors.fit(valid_embeddings)
    graph_parts = []
    edge_count = 0
    for start in range(0, len(valid_ids), batch_size):
        graph_part = neighbors.radius_neighbors_graph(
            valid_embeddings[start:start + batch_size], mode="distance"
        )
        edge_count += graph_part.nnz
        if edge_count > max_graph_edges:
            raise MemoryError(
                f"Radius graph exceeded {max_graph_edges:,} edges. Reduce EPS or "
                "increase MAX_GRAPH_EDGES if memory permits. No rows were sampled."
            )
        graph_parts.append(graph_part)

    graph = sparse.vstack(graph_parts, format="csr")
    del graph_parts
    # Explicit self-edges and off-diagonal zero distances must be retained.
    graph.setdiag(0.0)
    graph = sort_graph_by_row_values(graph, copy=False, warn_when_not_sorted=False)
    estimator = DBSCAN(eps=eps, min_samples=min_samples, metric="precomputed")
    labels[valid_ids] = estimator.fit_predict(graph, sample_weight=counts[valid_ids])
    return labels, estimator


unique_cluster_labels, dbscan_model = cluster_weighted_embeddings(reduced_embeddings, row_counts)
for df in datasets.values():
    df["cluster"] = unique_cluster_labels[df["embedding_row_id"].to_numpy()]

cluster_sizes = pd.Series(
    unique_cluster_labels[row_embedding_ids], name="cluster"
).value_counts().sort_index().rename("row_count")
print(f"Clusters: {(cluster_sizes.index >= 0).sum()}; noise rows: {cluster_sizes.get(-1, 0):,}")
display(cluster_sizes.to_frame())

**Average weighted embedding for each cluster**

For cluster C, `average_embedding = sum(row_count[i] * weighted_embeddings[i])
/ sum(row_count[i])`, using its unique feature combinations i. This is
exactly the mean over all original member rows from both datasets.
These means are not normalized again. The optional normalized copy is
provided separately for cosine comparisons.

In [ ]:
def average_cluster_embeddings(embeddings, labels, counts):
    valid_ids = np.flatnonzero(labels >= 0)
    members = valid_ids[np.argsort(labels[valid_ids], kind="stable")]
    cluster_ids, starts = np.unique(labels[members], return_index=True)
    means = np.empty((len(cluster_ids), embeddings.shape[1]), dtype=np.float32)
    sizes = np.zeros(len(cluster_ids), dtype=np.int64)
    for position, cluster_id in enumerate(cluster_ids):
        stop = starts[position + 1] if position + 1 < len(starts) else len(members)
        member_ids = members[starts[position]:stop]
        sizes[position] = counts[member_ids].sum()
        total = np.zeros(embeddings.shape[1], dtype=np.float64)
        for start in range(0, len(member_ids), 4096):
            ids = member_ids[start:start + 4096]
            total += np.einsum("i,ij->j", counts[ids], embeddings[ids], dtype=np.float64)
        means[position] = total / sizes[position]
    summary = pd.DataFrame({
        "cluster": cluster_ids,
        "row_count": sizes,
        "average_embedding": list(means),
    }).set_index("cluster")
    return summary, means


cluster_embeddings_df, cluster_embedding_matrix = average_cluster_embeddings(
    weighted_embeddings, unique_cluster_labels, row_counts
)
# Matrix row order follows cluster_embeddings_df.index, including when there are no clusters.
cluster_embedding_matrix_normalized = (
    normalize(cluster_embedding_matrix, norm="l2")
    if len(cluster_embedding_matrix) else cluster_embedding_matrix.copy()
)
for dataset_name, df in datasets.items():
    source_counts = df.loc[df["cluster"] >= 0, "cluster"].value_counts()
    cluster_embeddings_df[f"{dataset_name}_rows"] = source_counts.reindex(
        cluster_embeddings_df.index, fill_value=0
    ).astype(np.int64)

display(cluster_embeddings_df.head())
for dataset_name, df in datasets.items():
    print(dataset_name)
    product_column = "products" if "products" in df.columns else "product"
    display(df[[product_column, "adj", "verb", "cluster"]].head())

Results are available in both original dataframes (the four embedding
columns and `cluster`), and in `cluster_embeddings_df` (one average per
non-noise cluster). `cluster_embedding_matrix` contains the same averages
as a numeric matrix. For example,
`cluster_embeddings_df.loc[cluster_id, "average_embedding"]` retrieves a
cluster's weighted average.

References: [BERT model and feature extraction](https://huggingface.co/google-bert/bert-base-uncased),
[BERT hidden states](https://huggingface.co/docs/transformers/en/model_doc/bert),
and [DBSCAN parameters, duplicate weights, and sparse neighborhoods](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html).
[UMAP for clustering](https://umap-learn.readthedocs.io/en/latest/clustering.html).

**Recommendation architecture: a cluster-conditioned denoising autoencoder**

For each `(user, cluster)`, randomly hide 30% of the observed ratings on
each training pass. Encode only the remaining `(item embedding, rating)`
pairs, then reconstruct the hidden ratings using the latent user/cluster
representation, the target item's embedding, and the cluster mean.

```text
Visible rated items in ONE user/cluster group
      │ weighted BERT vectors + ratings + explicit observation mask
      ▼
Shared item projection (768 → 64) + rating encoder
      ▼
Masked mean pooling ─────── Cluster mean projection (768 → 64)
      └────────────────────┬───────────────────────────────┘
                           ▼
                 Encoder → latent vector (32)
                           │
            Target item vector + cluster projection
                           ▼
                   Decoder → sigmoid rating
                           ▼
                 Rescale to the rating scale
```

This is a **set denoising autoencoder**: a shared network accepts different
cluster sizes without creating a dense user-by-item matrix. It has no
learned item-ID lookup, so food items without ratings can be scored from
content. One model shares parameters across clusters; each example's
context and candidates stay within its own cluster. Food predictions are
transfer from recipe ratings and have no labeled food-rating validation.

Only randomly hidden **observed** ratings contribute to training MSE.
Unrated items are never assigned zero targets. Fixed validation ratings
are withheld before training and are excluded from every training input
and target. Validation is available for groups with at least three
ratings; at least two observations remain for masked training. Groups
with two ratings train without a validation holdout. Singletons can be
recommendation contexts but cannot train reconstruction from other items.
Noise `-1` is excluded, rather than treated as a cluster.

Training contexts are capped at 64 sampled ratings and masked targets
at 32 per group to bound memory. Validation reports RMSE/MAE on the
original rating scale and compares training-only mean baselines.
Validation covers eligible groups, not all users or unrated foods.
A user with no rated, non-noise cluster receives no personalized results.

In [ ]:
from cluster_recommender import (
    ClusterDenoisingAutoencoder,
    RatingCollator,
    build_catalog_and_ratings,
    split_rating_groups,
    train_autoencoder,
    validation_baselines,
    predict_unrated_items,
)

# Dataset policy: change RATING_MIN to 0 and ZERO_IS_MISSING to False if zero is a real rating.
ZERO_IS_MISSING = True
RATING_MIN, RATING_MAX = 1.0, 5.0
CANDIDATE_SOURCES = ("recipe", "food")
VALIDATION_FRACTION = 0.2
MASK_FRACTION = 0.3
LATENT_DIM = 32
AE_EPOCHS = 10
AE_BATCH_SIZE = 64
AE_LEARNING_RATE = 1e-3
MAX_CONTEXT_RATINGS = 64
MAX_MASKED_TARGETS = 32

torch.manual_seed(RANDOM_STATE)
item_catalog, observed_ratings = build_catalog_and_ratings(
    new_dataset_recommendation_df, current_dataset_df,
    rating_min=RATING_MIN, rating_max=RATING_MAX, zero_is_missing=ZERO_IS_MISSING,
)
training_groups, validation_pairs = split_rating_groups(
    observed_ratings, validation_fraction=VALIDATION_FRACTION, seed=RANDOM_STATE
)
clustered_ratings = observed_ratings.loc[observed_ratings["cluster"] >= 0]
coverage = pd.Series({
    "catalog_items": len(item_catalog),
    "observed_ratings": len(observed_ratings),
    "ratings_in_non_noise_clusters": len(clustered_ratings),
    "users_with_observed_ratings": observed_ratings["user_id"].nunique(),
    "users_with_rated_clusters": clustered_ratings["user_id"].nunique(),
    "training_user_cluster_groups": len(training_groups),
    "training_ratings": sum(len(group.items) for group in training_groups),
    "validation_ratings": sum(len(target.items) for _, target in validation_pairs),
}, name="count")
display(coverage.to_frame())
if not validation_pairs:
    print("No groups support held-out validation; training will retain the final epoch.")

In [ ]:
rating_collator = RatingCollator(weighted_embeddings, item_catalog, cluster_embeddings_df)
recommendation_model = ClusterDenoisingAutoencoder(
    embedding_dim=weighted_embeddings.shape[1], latent_dim=LATENT_DIM
)
print(recommendation_model)
training_history = train_autoencoder(
    recommendation_model, training_groups, validation_pairs, rating_collator, device,
    epochs=AE_EPOCHS, batch_size=AE_BATCH_SIZE, learning_rate=AE_LEARNING_RATE,
    mask_fraction=MASK_FRACTION, max_context=MAX_CONTEXT_RATINGS,
    max_targets=MAX_MASKED_TARGETS, rating_span=RATING_MAX - RATING_MIN,
    seed=RANDOM_STATE,
)
baseline_metrics = validation_baselines(
    training_groups, validation_pairs, rating_span=RATING_MAX - RATING_MIN
)
display(training_history)
display(baseline_metrics)
# The best validation epoch is restored. If validation is empty, the final epoch is retained.

**Predict the unrated items in a user's clusters**

At recommendation time, use all the user's genuinely known ratings as
context, including ratings previously set aside to measure validation.
This serving step is separate from validation. Predict every unrated
candidate from the configured sources in the user's rated clusters,
excluding already-rated items. Candidates with no ratings of their own
are supported through their weighted BERT features. Scores are estimated
ratings; no value is written back as an observed training label.

In [ ]:
# Choose an existing user for the example; replace with any user ID to personalize.
example_user_id = clustered_ratings["user_id"].value_counts().index[0]
predicted_unrated_ratings = predict_unrated_items(
    recommendation_model, example_user_id, item_catalog, observed_ratings,
    weighted_embeddings, cluster_embeddings_df, device,
    rating_min=RATING_MIN, rating_max=RATING_MAX,
    candidate_sources=CANDIDATE_SOURCES, max_context=MAX_CONTEXT_RATINGS,
    top_k=None, seed=RANDOM_STATE,
)
recommendations = predicted_unrated_ratings.head(20)
print(f"Predicted {len(predicted_unrated_ratings):,} unrated items for user {example_user_id}.")
display(recommendations)

The implementation is in `cluster_recommender.py`. The trained model is
`recommendation_model`; metrics are in `training_history` and
`baseline_metrics`; `predicted_unrated_ratings` contains scores for the
example user's unrated candidates, and `recommendations` shows the top 20.
Change `example_user_id` or call `predict_unrated_items` for another user.
These cells operate on the full loaded datasets; no silent row sampling is used.